### Latent Factor Vasicek Model
Considering a homogenous risk class, every oligor $i$ in the class has the following asset return at time $t$:
$$Z_{i,t} = \sqrt{\rho} \cdot X_t + \sqrt{1-\rho} \cdot \varepsilon_{i,t}$$

where:
- $X_t \sim N(0,1)$ is the systemic (market/economy) factor - this impacts all obligors
- $\varepsilon_{i,t} \sim N(0,1)$ is the idiosyncratic risk (specific to obligor $i$)
- $\rho \in [0,1]$ is the factor loading (squared because we assume N(0,1)) - asset correlation, **(assumed constant here, obviously a problem)**
- Obligor $i$ defaults when $Z_{i,t} < \Phi^{-1}(PD_i)$ (standardized return of the asset falls below a specific threshold of uncond PD)
> I mentioned the temporal component here to highlight the constant $\rho$ assumption. 

Each obligor has a latent variable ($Z_i$, representing firm value) driven by the common systemic factor $X$ and an idiosyncratic factor $\varepsilon_i$.

### Conditional PD
Given the realization of the systemic factor $X = x$, an obligor's probability of default is:
$$PD_i(x) = \Phi\!\left(\frac{\Phi^{-1}(PD_i) - \sqrt{\rho_i}\, x}{\sqrt{1 - \rho_i}}\right)$$


>  The idea is that as the portfolio becomes large enough (such that no single exposure has a dominating proportion) the portfolio loss, *conditional on the factor*, becomes deterministic.

### Gaussian Copula 
The Gaussian copula couples marginal $PD$ distributions using the multivariate normal:
$$C(u_1, \ldots, u_n) = \Phi_n\!\left(\Phi^{-1}(u_1), \ldots, \Phi^{-1}(u_n); R\right)$$

where $R$ is the correlation matrix, $\Phi_n$ is the $n$-dimensional N(0,1) cdf and $u_i = F_i(z_i)$ are the marginal CDFs.

In [32]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats, optimize
from scipy.special import ndtri  # (inverse N(0,1) CDF)

np.random.seed(123)

---
## 1. Building the Default Rate 

The default rate of a cluster $k$ can be defined as:
$$\mu_k = \frac{\text{Number of defaults in cluster } k}{\text{Number of alive loans in cluster } k}$$

> $\mu_k$ is backwards-looking, but it can be used to estimate the (forward-looking) $PD_k$. 

In [33]:
df = pd.read_parquet('lending_club_subset.parquet')
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,home_ownership,annual_inc,issue_d,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high
0,3600.0,36 months,13.99,C,C4,MORTGAGE,55000.0,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0
1,24700.0,36 months,11.99,C,C1,MORTGAGE,65000.0,Dec-2015,Fully Paid,small_business,SD,16.06,715.0,719.0
2,20000.0,60 months,10.78,B,B4,MORTGAGE,63000.0,Dec-2015,Fully Paid,home_improvement,IL,10.78,695.0,699.0
3,35000.0,60 months,14.85,C,C5,MORTGAGE,110000.0,Dec-2015,Current,debt_consolidation,NJ,17.06,785.0,789.0
4,10400.0,60 months,22.45,F,F1,MORTGAGE,104433.0,Dec-2015,Fully Paid,major_purchase,PA,25.37,695.0,699.0


In [34]:
status_counts = df['loan_status'].value_counts() #distribution of loan statuses
for status, count in status_counts.items():
    print(f" {status:<55s} {count:>8,d} ({count/len(df):4.3%})")

 Fully Paid                                              1,076,751 (47.629%)
 Current                                                  878,317 (38.852%)
 Charged Off                                              268,559 (11.879%)
 Late (31-120 days)                                        21,467 (0.950%)
 In Grace Period                                            8,436 (0.373%)
 Late (16-30 days)                                          4,349 (0.192%)
 Does not meet the credit policy. Status:Fully Paid         1,988 (0.088%)
 Does not meet the credit policy. Status:Charged Off          761 (0.034%)
 Default                                                       40 (0.002%)


In [35]:
#  default rate = defaulted/alive loans 

# Default can have different definitions 
# I exclude grace period & short-term late from defaulted statuses because they aren't necessarily defaults (yet, at least)
defaulted_statuses = {
    'Charged Off',
    'Default',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
}

alive_statuses = {
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid',
    'Current'
}

default_or_alive_mask = df['loan_status'].isin(defaulted_statuses | alive_statuses)
df_filtered=df.loc[default_or_alive_mask]

df_filtered['is_default'] = df_filtered['loan_status'].isin(defaulted_statuses).astype(int)
# 1=default, 0=alive

default_count = df_filtered['is_default'].sum()
alive_count = (df_filtered['is_default'] == 0).sum()
default_rate = default_count / alive_count

print(f"Defaults:    {default_count:,}")
print(f"Alive loans: {alive_count:,}")
print(f"Default rate: {default_rate:.3%}")

Defaults:    290,827
Alive loans: 1,957,056
Default rate: 14.860%


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/143548615.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Default rates by grade. I also include average loan amount and average FICO score by grade since I'll use these stats to help interpret the copula results down the line.

In [36]:
grade_stats = (df_filtered.groupby('grade').agg(
    n_total=('is_default', 'size'),
    n_defaults=('is_default', 'sum'),
    avg_loan=('loan_amnt', 'mean'),
    avg_fico=('fico_range_low', 'mean'),
).sort_index())

grade_stats['n_alive'] = grade_stats['n_total'] - grade_stats['n_defaults']
grade_stats['pd_empirical'] = grade_stats['n_defaults'] / grade_stats['n_alive']  # default rate 

print(f"{'Grade':<8s} {'# Alive':>10s} {'# Defaults':>12s} {'Default Rate':>16s} {'Avg Loan Amt':>12s} {'Avg FICO':>10s}")
for grade, row in grade_stats.iterrows():
    print(f"  {grade:<6s} {row['n_alive']:>10,.0f} {row['n_defaults']:>12,.0f} "
          f"{row['pd_empirical']:>16.2%} {row['avg_loan']:>9,.0f} {row['avg_fico']:>10.0f}")
print(f"  {'Total':<6s} {grade_stats['n_alive'].sum():>10,.0f} "
      f"{grade_stats['n_defaults'].sum():>12,.0f} "
      f"{(grade_stats['n_defaults'].sum() / grade_stats['n_alive'].sum()):>16.2%}")

Grade       # Alive   # Defaults     Default Rate Avg Loan Amt   Avg FICO
  A         416,518       15,536            3.73%    14,599        729
  B         603,344       57,449            9.52%    14,164        700
  C         552,225       93,359           16.91%    15,021        689
  D         255,531       66,025           25.84%    15,693        684
  E          96,073       38,365           39.93%    17,441        682
  F          26,206       15,225           58.10%    19,106        680
  G           7,159        4,868           68.00%    20,361        679
  Total   1,957,056      290,827           14.86%


In [37]:
# default rate visualization
colors = ['#2ecc71', '#27ae60', '#f1c40f', '#e67e22', '#e74c3c', '#c0392b', '#8e44ad']
total_loans = grade_stats['n_total']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Default Rate by Grade", "Number of Loans by Grade"),
    horizontal_spacing=0.12
)

# Left panel: default rate
fig.add_trace(
    go.Bar(
        x=grade_stats.index,
        y=grade_stats['pd_empirical'],
        marker_color=colors[:len(grade_stats)],
        text=[f"{pd:.1%}" for pd in grade_stats['pd_empirical']],
        textposition='outside',
        name='Default Rate'
    ), row=1, col=1
)

# Right panel: total loan counts
fig.add_trace(
    go.Bar(
        x=grade_stats.index,
        y=total_loans,
        marker_color=colors[:len(grade_stats)],
        text=[f"{n:,.0f}" for n in total_loans],
        textposition='outside',
        name='# Loans'
    ), row=1, col=2
)

fig.update_yaxes(title_text="Default Rate", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="Number of Loans", row=1, col=2)
fig.update_layout(
    template="plotly_white", showlegend=False, height=450
)
fig.show()

- **Grade A** has the lowest default rate, **Grade G** the highest, as one would expect
- The left tail (clusters F, G) have fewer observations

Each grade defines a natural **cluster** of obligors with similar creditworthiness. I will use these clusters in building the copula.

---
## 2. Copula & Clustering Obligors

(We will not consider the temporal component here)

In a large portfolio (say,$n = 1{,}000{,}000+$ loans), modeling every pairwise default correlation is computationally unfeasible, so we make some simplifying assumptions and group obligors into **$K$ clusters** (i.e, the 7 Lending Club grades A–G).

We assume that:

1. **Within a cluster**, all obligors share the same marginal default probability $PD_k$ and factor loading $\sqrt{\rho_k}$ (the latter controls how strongly each cluster's $PD$ responds to $X$)
2. **Across clusters**, dependence is driven by the **common systemic factor** $X$

### The Gaussian Copula for Clustered Obligors

Let $D_k$ be the number of defaults in cluster $k$ (with $n_k$ obligors). Using the **Gaussian copula** for the joint default distribution works as follows:

**Step 1: Vasicek model for each obligor**

For obligor $i$ in cluster $k$:
$$Z_{k,i} = \sqrt{\rho_k}\, X + \sqrt{1 - \rho_k}\, \varepsilon_{k,i}$$

where $X, \varepsilon_{k,i} \stackrel{\text{iid}}{\sim} N(0,1)$. 
> **($ X,\varepsilon \sim N$  is a pretty strong assumption)**

**Step 2: Default threshold**

Obligor $i$ in cluster $k$ defaults if:
$$Z_{k,i} < \Phi^{-1}(PD_k) \equiv c_k$$

**Step 3: Conditional independence**

Given $X = x$, defaults within each cluster are **independent Bernoulli** trials (summing up to a **Binomial**):
$$D_k \mid X = x \sim \text{Binomial}\!\left(n_k,\; \Phi\!\left(\frac{c_k - \sqrt{\rho_k}\, x}{\sqrt{1-\rho_k}}\right)\right)$$

They are independent trials because we assume that dependence is based solely on $X$.

**Step 4: Cluster level representation**

The joint distribution of default counts across all $K$ clusters is obtained by integrating out $X$ (integrating over the distribution of the systemic factor):
$$P(D_1 \leq d_1, \ldots, D_K \leq d_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} P(D_k \leq d_k \mid X = x)\;\phi(x)\,dx$$

where $\phi$ is the standard normal density. We integrate over the distribution of $X$ in order to "average" over all possible values of it, weighted by their density.

**Step 5: Gaussian Copula**

Given $K$ clusters, let $D_k \in {0, 1, \dots, n_k}$ be the default count in cluster $k$. The joint distribution of $(D_1, \dots, D_K)$ is driven by a Gaussian copula with a block-correlation structure induced by the one-factor model.

The Gaussian copula couples the marginal default distributions across the $K$ clusters using the multivariate normal:

$$C(u_1, \ldots, u_K) = \Phi_K\!\left(\Phi^{-1}(u_1), \ldots, \Phi^{-1}(u_K); \mathbf{R}\right)$$

where $\mathbf{R}$ is the $K \times K$ cluster correlation matrix with elements 
$R_{km} = \sqrt{\rho_k \rho_m}$ for $k \neq m$, $\Phi_K$ is the $K$-dimensional standard normal CDF, 
and $u_k = F_k(d_k)$ is the marginal CDF of the default count for cluster $k$.

Under the one-factor structure, this reduces to a one-dimensional integral:

$$\Phi_K\!\left(\Phi^{-1}(u_1), \ldots, \Phi^{-1}(u_K); \mathbf{R}\right) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \Phi\!\left(\frac{\Phi^{-1}(u_k) - \sqrt{\rho_k}\, x}{\sqrt{1 - \rho_k}}\right) \phi(x)\, dx$$

In [38]:
grades = grade_stats.index.tolist()       # [A-G]
K = len(grades)
pds = grade_stats['pd_empirical'].values  # empirical PD (default rates, defaults/alive)
n_k = grade_stats['n_alive'].values       # cluster size (alive loans)
thresholds = ndtri(pds)                   # percentile func of N(0,1)

print(f"{'Grade':<8s} {'n_k (alive)':>12s} {'PD_k':>10s} {'c_k = phi⁻¹(PD)':>16s}")
for i, g in enumerate(grades):
    print(f"  {g:<6s} {n_k[i]:>12,d} {pds[i]:>10.4f} {thresholds[i]:>16.4f}")

Grade     n_k (alive)       PD_k  c_k = phi⁻¹(PD)
  A           416,518     0.0373          -1.7829
  B           603,344     0.0952          -1.3093
  C           552,225     0.1691          -0.9579
  D           255,531     0.2584          -0.6483
  E            96,073     0.3993          -0.2551
  F            26,206     0.5810           0.2044
  G             7,159     0.6800           0.4677


The thresholds $c_k$ are growing as grade worsens.

So an economy shock doesn't have to be too extreme to trigger default for grades with poor creditworthiness.

## 3. Solving for Asset Correlations

While the marginal default probabilities $PD_k$ are observed from historical data, the asset correlations $\rho_k$ (and thus the factor loadings $\sqrt{\rho_k}$) are latent and must be estimated.

### Moment Matching via Pairwise Default Correlation

For a homogeneous cluster $k$, the relationship between asset correlation $\rho_k$ and pairwise default correlation $\rho_k^{\text{def}}$ is:

$$\rho_k^{\text{def}} = \frac{\Phi_2\!\left(\Phi^{-1}(PD_k), \Phi^{-1}(PD_k); \rho_k\right) - PD_k^2}{PD_k(1 - PD_k)}$$

where $\Phi_2(\cdot, \cdot; \rho)$ is the bivariate standard normal CDF with correlation $\rho$.
Because we assume independence between defaults conditioned on $X$, we can say just do pairwsie default correlation.

If an empirical estimate of the default correlation $\hat{\rho}_k^{\text{def}}$ is available (e.g., from historical co-default frequencies), $\rho_k$ can be recovered by inverting this monotonic relationship.

### Moment Matching via Variance of Default Rates

Alternatively, the asset correlation can be inferred from the time-series variance of the observed default rate $p_k(X)$:

$$\text{Var}[p_k(X)] = \mathbb{E}[p_k(X)^2] - PD_k^2$$

Using properties of the Gaussian copula, the expected squared conditional default probability is:

$$\mathbb{E}[p_k(X)^2] = \Phi_2\!\left(\Phi^{-1}(PD_k), \Phi^{-1}(PD_k); \rho_k\right)$$

Hence, the default correlation satisfies:

$$\rho_k^{\text{def}} = \frac{\text{Var}[p_k(X)]}{PD_k(1 - PD_k)}$$

Since $\rho_k^{\text{def}}$ is strictly increasing in $\rho_k$ (for $\rho_k \geq 0$), the inverse mapping is well-defined and can be solved numerically via Brent's method.

In [39]:
# Time-series moment matching 
# I parse the issue date and compute default rates by grade and quarter
df_filtered['issue_d'] = pd.to_datetime(df_filtered['issue_d'], format='%b-%Y')
df_filtered['quarter'] = df_filtered['issue_d'].dt.to_period('Q')

# Default rate by grade & quarter
ts = (
    df_filtered.groupby(['grade', 'quarter'])
    .agg(n=('is_default', 'size'), defaults=('is_default', 'sum'))
    .reset_index()
)
ts['n_alive'] = ts['n'] - ts['defaults']
ts['pd_quarterly'] = ts['defaults'] / ts['n_alive']

# keep the quarters that have enough data (at least 50 alive loans per grade)
ts = ts[ts['n_alive'] >= 50]

print(f"{ts['quarter'].nunique()} quarters × {ts['grade'].nunique()} grades")
print(f"Quarter range: {ts['quarter'].min()} to {ts['quarter'].max()}")

45 quarters × 7 grades
Quarter range: 2007Q4 to 2018Q4


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/3270199279.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/3270199279.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [40]:
from scipy.stats import norm, multivariate_normal
def invert_default_to_asset_correlation(pd_val, rho_default, tol=1e-8):
    """
    Invert observed default correlation (from conditional PD variance) to latent asset correlation
    using the one-factor Gaussian copula model.
    
    Args:
        pd_val (float): Marginal default probability (PD) of the grade.
        rho_default (float): Observed default correlation (from Var[conditional PD]).
        tol (float): Tolerance for root-finding.
    
    Returns:
        rho_asset (float): Latent asset correlation (factor loading squared).
    """
    c = norm.ppf(pd_val)  # Convert PD to normal quantile
    
    # Target value for BVN CDF corresponding to observed default correlation
    target = rho_default * pd_val * (1 - pd_val) + pd_val**2

    def objective(rho_asset):
        """
        Difference between model BVN CDF and target
        f(rho_asset) = 0 at the correct latent correlation
        """
        # Ensure rho_asset is within (0,1) to avoid singular covariance
        rho_asset = np.clip(rho_asset, 1e-12, 1-1e-12)
        
        cov = np.array([[1, rho_asset],
                        [rho_asset, 1]])
        bvn_cdf = multivariate_normal.cdf([c, c], mean=[0, 0], cov=cov)
        return bvn_cdf - target

    try:
        # Brent's method to find the root in [0,1]
        rho_asset = optimize.brentq(objective, 0.0, 0.999, xtol=tol)
    except ValueError:
        # If root-finding fails return NaN
        rho_asset = np.nan
    
    return rho_asset

# example: compute asset correlations from quarterly PD time series
rho_ts = {}
for i, g in enumerate(grades):
    ts_g = ts[ts['grade'] == g]['pd_quarterly']
    var_pd = ts_g.var()
    pd_val = pds[i]
    
    # Step 1: Compute default correlation from conditional PD variance
    rho_def = var_pd / (pd_val * (1 - pd_val))
    rho_def = np.clip(rho_def, 0.001, 0.999)  # avoid extreme values
    
    # Step 2: Invert to latent asset correlation
    rho_asset = invert_default_to_asset_correlation(pd_val, rho_def)
    
    rho_ts[g] = {'rho_default': rho_def, 'rho_asset': rho_asset, 'var_pd': var_pd}

rho_moment = np.array([rho_ts[g]['rho_asset'] for g in grades])

# Print summary
print("Moment-Matching Asset Correlations (from time-series variance)")
print(f"{'Grade':<8s} {'Var[p(X)]':>12s} {'ρ_default':>12s} {'ρ_asset':>12s}")
for i, g in enumerate(grades):
    print(f"  {g:<6s} {rho_ts[g]['var_pd']:>12.6f} {rho_ts[g]['rho_default']:>12.4f} "
          f"{rho_moment[i]:>12.4f}")

Moment-Matching Asset Correlations (from time-series variance)
Grade       Var[p(X)]    ρ_default      ρ_asset
  A          0.000453       0.0126       0.0621
  B          0.002299       0.0267       0.0753
  C          0.004692       0.0334       0.0714
  D          0.008926       0.0466       0.0839
  E          0.017183       0.0716       0.1146
  F          0.039010       0.1602       0.2517
  G          0.076429       0.3512       0.5407


In [41]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=grades, y=rho_moment,
    mode='lines+markers',
    line=dict(color='#e74c3c', width=3), marker=dict(size=10)
))

fig.update_layout(
    title="Asset Correlation Estimates: Moment Matching (Time-Series)",
    xaxis_title="Grade",
    yaxis_title="Asset Correlation (ρ)",
    template="plotly_white", height=420,
    legend=dict(x=0.6, y=0.95)
)
fig.show()

In [42]:
# Choose final factor loadings 
# We use moment matching estimates; replace NaN with a conservative default (0.10)
rho_final = np.where(np.isnan(rho_moment), 0.10, rho_moment)
sqrt_rho = np.sqrt(rho_final)  # factor loadings

print("Final Factor Loadings (√ρ)")
for i, g in enumerate(grades):
    print(f" For  Grade {g}: ρ = {rho_final[i]:.4f}  →  √ρ = {sqrt_rho[i]:.4f}")

Final Factor Loadings (√ρ)
 For  Grade A: ρ = 0.0621  →  √ρ = 0.2491
 For  Grade B: ρ = 0.0753  →  √ρ = 0.2744
 For  Grade C: ρ = 0.0714  →  √ρ = 0.2673
 For  Grade D: ρ = 0.0839  →  √ρ = 0.2896
 For  Grade E: ρ = 0.1146  →  √ρ = 0.3385
 For  Grade F: ρ = 0.2517  →  √ρ = 0.5017
 For  Grade G: ρ = 0.5407  →  √ρ = 0.7353


## Alternative - Estimation of $\rho_k$ via MLE

Instead of matching moments from time-series variance, we directly maximize the likelihood of the observed default data under the one-factor Gaussian copula model.

Let $d_k$ be the number of defaults observed in cluster $k$, out of $n_k$ obligors. The conditional likelihood given $X=x$ is:

$$\mathcal{L}(x; \boldsymbol{\rho}) = \prod_{k=1}^{K} \binom{n_k}{d_k} \, p_k(x)^{d_k}\,(1-p_k(x))^{n_k - d_k}$$

Integrating out the latent factor $X$ yields the unconditional likelihood:

$$\mathcal{L}(\boldsymbol{\rho}) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \binom{n_k}{d_k} \, p_k(x)^{d_k}\,(1-p_k(x))^{n_k - d_k} \, \phi(x)\, dx$$

The log-likelihood is:

$$\ell(\boldsymbol{\rho}) = \log \int_{-\infty}^{\infty} \prod_{k=1}^{K} \left[ p_k(x)^{d_k} (1-p_k(x))^{n_k - d_k} \right] \phi(x)\, dx + \text{const.}$$

and the Binomial coefficients $\binom{n_k}{d_k}$ are absorbed into the constant term, as they do not depend on $\boldsymbol{\rho}$.

The integral over $X$ is evaluated via Gauss--Hermite quadrature (50-point), and we implement JAX for automatic differentiation with L-BFGS optimization.

In [43]:
import jax
import jax.numpy as jnp
import jax_mle as jax_mle
from jax_mle import (
    fit_rho_bfgs,
    log_likelihood,
    get_gh_nodes_weights,
    cond_default_prob,
)

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX version: {jax.__version__}")

JAX backend: cpu
JAX version: 0.8.2


In [44]:
# Build D (default counts) and N_obligors from quarterly time-series
# Each row = one quarter, K columns = grades

ts_pivot_d = ts.pivot_table(index='quarter', columns='grade', values='defaults', fill_value=0)
ts_pivot_n = ts.pivot_table(index='quarter', columns='grade', values='n', fill_value=0)

# Ensure columns are in grade order
ts_pivot_d = ts_pivot_d.reindex(columns=grades, fill_value=0)
ts_pivot_n = ts_pivot_n.reindex(columns=grades, fill_value=0)

# Keep only quarters with obligors in ALL grades (or adjust as needed)
valid_mask = (ts_pivot_n > 0).all(axis=1)

D = jnp.array(ts_pivot_d.loc[valid_mask].values.astype(int))
N_obligors = jnp.array(ts_pivot_n.loc[valid_mask].values.astype(int))

# Sanity check
assert (D <= N_obligors).all(), "Defaults exceed obligors in some cell!"

print(f"Default count matrix D: {D.shape}  (N_quarters × K)")
print(f"Obligor count matrix N: {N_obligors.shape}")
print(f"\nQuarters retained: {valid_mask.sum()} / {len(ts_pivot_d)}")
print(f"\nDefault counts per grade (mean across retained quarters):")
for k, g in enumerate(grades):
    emp_pd = D[:, k].sum() / N_obligors[:, k].sum()
    print(f"  Grade {g}: avg defaults = {D[:, k].mean():.1f} / {N_obligors[:, k].mean():.0f}"
          f"  (rate = {emp_pd:.4f}, input PD = {pds[k]:.4f})")

Default count matrix D: (24, 7)  (N_quarters × K)
Obligor count matrix N: (24, 7)

Quarters retained: 24 / 45

Default counts per grade (mean across retained quarters):
  Grade A: avg defaults = 591.0 / 17115  (rate = 0.0345, input PD = 0.0373)
  Grade B: avg defaults = 2226.8 / 26147  (rate = 0.0852, input PD = 0.0952)
  Grade C: avg defaults = 3716.2 / 25917  (rate = 0.1434, input PD = 0.1691)
  Grade D: avg defaults = 2618.5 / 12814  (rate = 0.2044, input PD = 0.2584)
  Grade E: avg defaults = 1519.0 / 5317  (rate = 0.2857, input PD = 0.3993)
  Grade F: avg defaults = 595.4 / 1612  (rate = 0.3693, input PD = 0.5810)
  Grade G: avg defaults = 191.3 / 470  (rate = 0.4070, input PD = 0.6800)


Let $P$ be the joint distribution of the whole sequence $(X_i){i=-\infty}^{\infty}$.
Let $\mathcal{F}{-\infty}^t$ be the $\sigma$-algebra generated by the past up to $t$.
Let $\mathcal{F}_{t+n}^{\infty}$ be the $\sigma$-algebra generated by the future from $t+n$.

exponentially $\beta$-mixing (i.e., $\beta(n) \leq C \cdot e^{-bn}$)

In [45]:
# Fit asset correlations via MLE (L-BFGS + JAX autodiff)

# Keep counts as integers, PDs and ρ as float64
D_jax = jnp.array(D, dtype=jnp.int32)
N_jax = jnp.array(N_obligors, dtype=jnp.int32)
m_pd_vec = jnp.array(pds, dtype=jnp.float64)

# Initial guess -use moment-matching estimates 
if 'rho_moment' in locals():
    init_rho = jnp.array(np.where(np.isnan(rho_moment), 0.10, rho_moment), dtype=jnp.float64)
else:
    init_rho = jnp.full(len(grades), 0.10, dtype=jnp.float64)

init_rho = jnp.clip(init_rho, 0.01, 0.99)

print("Running MLE via L-BFGS (JAX)...")
print(f"  Initial asset correlations: {np.array(init_rho).round(4)}")
print(f"  Marginal PDs: {np.array(m_pd_vec).round(4)}")

# Sanity check
init_ll = log_likelihood(init_rho, D_jax, N_jax, m_pd_vec)
print(f"  Initial log-likelihood: {init_ll:.4f}")
assert jnp.isfinite(init_ll), "Initial likelihood is NaN/Inf!"

rho_mle = fit_rho_bfgs(D_jax, N_jax, m_pd_vec, init_rho)
rho_mle = np.array(rho_mle)

print(f"\nMLE Asset Correlations (ρ_k):")
for k, g in enumerate(grades):
    print(f"  Grade {g}: ρ = {rho_mle[k]:.4f}  →  √ρ = {np.sqrt(rho_mle[k]):.4f}")

Running MLE via L-BFGS (JAX)...
  Initial asset correlations: [0.0621 0.0753 0.0714 0.0839 0.1146 0.2517 0.5407]
  Marginal PDs: [0.0373 0.0952 0.1691 0.2584 0.3993 0.581  0.68  ]
  Initial log-likelihood: -10480.9531


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/1887528641.py:6: UserWarning:

Explicitly requested dtype float64 requested in array is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.

/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/1887528641.py:10: UserWarning:

Explicitly requested dtype float64 requested in array is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.




MLE Asset Correlations (ρ_k):
  Grade A: ρ = 0.0069  →  √ρ = 0.0828
  Grade B: ρ = 0.0097  →  √ρ = 0.0987
  Grade C: ρ = 0.0140  →  √ρ = 0.1183
  Grade D: ρ = 0.0181  →  √ρ = 0.1345
  Grade E: ρ = 0.0294  →  √ρ = 0.1716
  Grade F: ρ = 0.0618  →  √ρ = 0.2487
  Grade G: ρ = 0.0845  →  √ρ = 0.2906


In [46]:
#comparison - MLE vs moment-matching estimates
# Convert MLE results to numpy
rho_mle_arr = np.array(rho_mle)  # fit_rho_bfgs returns asset correlations ρ
sqrt_rho_mle = np.sqrt(rho_mle_arr)

# Moment-matching estimates 
rho_moment = rho_final 

print(f"\n{'Grade':<8} {'ρ (moment)':>12} {'ρ (MLE)':>12} {'√ρ (moment)':>14} {'√ρ (MLE)':>12}")
print("-" * 60)
for k, g in enumerate(grades):
    print(f"{g:<8} {rho_moment[k]:>12.4f} {rho_mle_arr[k]:>12.4f} "
          f"{np.sqrt(rho_moment[k]):>14.4f} {sqrt_rho_mle[k]:>12.4f}")

# Log-likelihood comparison 
ll_moment = float(log_likelihood(
    jnp.array(rho_moment, dtype=jnp.float64), 
    D_jax, N_jax, 
    jnp.array(pds, dtype=jnp.float64)
))
ll_mle = float(log_likelihood(
    jnp.array(rho_mle_arr, dtype=jnp.float64), 
    D_jax, N_jax, 
    jnp.array(pds, dtype=jnp.float64)
))

print(f"\nLog-likelihood (moment-matching): {ll_moment:,.2f}")
print(f"Log-likelihood (MLE):             {ll_mle:,.2f}")
print(f"Improvement:                      {ll_mle - ll_moment:+,.2f}")


Grade      ρ (moment)      ρ (MLE)    √ρ (moment)     √ρ (MLE)
------------------------------------------------------------
A              0.0621       0.0069         0.2491       0.0828
B              0.0753       0.0097         0.2744       0.0987
C              0.0714       0.0140         0.2673       0.1183
D              0.0839       0.0181         0.2896       0.1345
E              0.1146       0.0294         0.3385       0.1716
F              0.2517       0.0618         0.5017       0.2487
G              0.5407       0.0845         0.7353       0.2906

Log-likelihood (moment-matching): -10,480.95
Log-likelihood (MLE):             -7,129.09
Improvement:                      +3,351.87


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/820259483.py:17: UserWarning:

Explicitly requested dtype float64 requested in array is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.

/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/820259483.py:19: UserWarning:

Explicitly requested dtype float64 requested in array is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.

/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_11170/820259483.py:22: UserWarning:

Explicitly requested dtype float64 requested in array is not available, and will be truncated to dtype float32. To enable more dtype

In [47]:
# Visualization: MLE vs moment-matching rho
# added Basel just because I thought it would be interesting
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=grades, y=rho_moment,
    mode='lines+markers', name='Moment Matching',
    line=dict(color='#e74c3c', width=3), marker=dict(size=10)
))
fig.add_trace(go.Scatter(
    x=grades, y=rho_mle_arr,
    mode='lines+markers', name='MLE (JAX)',
    line=dict(color='#2980b9', width=3, dash='dash'), marker=dict(size=10, symbol='diamond')
))

fig.add_hrect(y0=0.12, y1=0.24, line_width=0, fillcolor="gray", opacity=0.1,
              annotation_text="Basel II typical range", annotation_position="top right")

fig.update_layout(
    title="Asset Correlation Estimates: Moment Matching vs MLE",
    xaxis_title="Grade",
    yaxis_title="Asset Correlation (ρ)",
    template="plotly_white", height=450,
    legend=dict(x=0.6, y=0.95),
    xaxis=dict(categoryorder='array', categoryarray=grades)
)

fig.show()

## 4. Joint Default Probability Distribution

Using the MLE-estimated asset correlations $\hat{\rho}_k$, the joint CDF of default counts across the $K$ clusters is:

$$P(D_1 \leq d_1, \ldots, D_K \leq d_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \left[ \sum_{j=0}^{d_k} \binom{n_k}{j}\, p_k(x)^j\,(1 - p_k(x))^{n_k - j} \right] \phi(x)\,dx$$

where:

- $D_k$ is the random variable for defaults in cluster $k$
- $d_k$ is the observed (or threshold) default count
- $n_k$ is the number of obligors in cluster $k$
### Explanation of Terms

| Element | Purpose |
| :--- | :--- |
| $\binom{n_k}{j}\, p_k(x)^j\,(1 - p_k(x))^{n_k - j}$ | **Binomial PMF:** probability of exactly $j$ defaults in cluster $k$ given $X=x$ |
| $\sum_{j=0}^{d_k} (\cdot)$ | **Sum:** accumulates probabilities for $0, 1, \dots, d_k$ defaults (CDF of cluster $k$) |
| $\prod_{k=1}^{K} (\cdot)$ | **Product:** combines clusters via conditional independence given $X=x$ |
| $\int_{-\infty}^{\infty} (\cdot)\,\phi(x)\,dx$ | **Integral:** averages over the latent systemic factor $X \sim \mathcal{N}(0,1)$ |

- **Sum:** Defaults are discrete counts, so $P(D_k \leq d_k)$ requires summing the Binomial PMF.
- **Product:** Conditional on $X$, all dependence is captured—clusters are independent.
- **Integral:** $X$ is unobserved; we integrate it out to obtain the unconditional joint distribution.

In [48]:
# Joint distribution parameters - MLE

rho_mle_final = rho_mle_arr.copy()          # rho_k from MLE
sqrt_rho_mle_final = sqrt_rho_mle.copy()    # loadings)
thresholds_mle = ndtri(pds)                 

n_k_total = N_obligors.sum(axis=0)

print("\nJoint Distribution — MLE-Parameterized Components")
print(f"{'Grade':<8s} {'n_k':>10s} {'PD_k':>10s} {'c_k':>10s} "
      f"{'√ρ̂_k':>12s} {'ρ̂_k (MLE)':>12s}")
for k, g in enumerate(grades):
    print(f"  {g:<6s} {n_k_total[k]:>10,d} {pds[k]:>10.4f} {thresholds_mle[k]:>10.4f} "
          f"{sqrt_rho_mle_final[k]:>12.4f} {rho_mle_final[k]:>12.4f}")


Joint Distribution — MLE-Parameterized Components
Grade           n_k       PD_k        c_k        √ρ̂_k   ρ̂_k (MLE)
  A         410,756     0.0373    -1.7829       0.0828       0.0069
  B         627,519     0.0952    -1.3093       0.0987       0.0097
  C         622,015     0.1691    -0.9579       0.1183       0.0140
  D         307,525     0.2584    -0.6483       0.1345       0.0181
  E         127,617     0.3993    -0.2551       0.1716       0.0294
  F          38,697     0.5810     0.2044       0.2487       0.0618
  G          11,279     0.6800     0.4677       0.2906       0.0845


In [49]:
# Conditional PD curves using MLE asset correlations
def conditional_pd_mle(x, pd_marginal, rho):
    """p_k(x) = Φ( (Φ⁻¹(PD_k) - √ρ_k·x) / √(1 - ρ_k) )"""
    c = ndtri(pd_marginal)
    return norm.cdf((c - np.sqrt(rho) * x) / np.sqrt(1 - rho))

x_grid = np.linspace(-4, 4, 500)

fig = go.Figure()
for k, g in enumerate(grades):
    cpd_mle = conditional_pd_mle(x_grid, pds[k], rho_mle_final[k])
    fig.add_trace(go.Scatter(
        x=x_grid, y=cpd_mle, mode='lines',
        name=f'Grade {g}  (PD={pds[k]:.1%}, ρ̂={rho_mle_final[k]:.3f})',
        line=dict(width=2.5)
    ))

fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="Normal (X=0)")
fig.add_vline(x=-2, line_dash="dot", line_color="red", annotation_text="Bad (X=−2)")
fig.add_vline(x=-3, line_dash="dot", line_color="darkred", annotation_text="Severe (X=−3)")

fig.update_layout(
    title="Conditional PD p_k(x) vs Systematic Factor X  [MLE Estimates]",
    xaxis_title="Systematic Factor X  (← recession | expansion →)",
    yaxis_title="Conditional Default Probability",
    yaxis_tickformat=".0%",
    template="plotly_white", height=500,
    legend=dict(x=0.70, y=0.95)
)
fig.show()

In [50]:
# Scenario table: conditional default rates - MLE loadings
print("Conditional Default Rates by Scenario  [MLE Factor Loadings]")
x_scenarios = {'Good (+2σ)': 2, 'Ok (+1σ)': 1, 'Normal (0)': 0,
               'Recession (−1σ)': -1, 'Bad (−2σ)': -2, 'Severe (−3σ)': -3}

header = f"{'Scenario':<20s}" + "".join(f"{'Grade ' + g:>10s}" for g in grades)
print(header)
for scenario_name, x_val in x_scenarios.items():
    row = f"  {scenario_name:<18s}"
    for k in range(K):
        cpd = conditional_pd_mle(x_val, pds[k], sqrt_rho_mle_final[k])
        row += f"{cpd:>10.2%}"
    print(row)

Conditional Default Rates by Scenario  [MLE Factor Loadings]
Scenario               Grade A   Grade B   Grade C   Grade D   Grade E   Grade F   Grade G
  Good (+2σ)             0.69%     2.06%     3.98%     6.87%    11.69%    18.01%    23.43%
  Ok (+1σ)               1.53%     4.36%     8.28%    13.76%    23.10%    36.71%    46.62%
  Normal (0)             3.13%     8.39%    15.38%    24.29%    38.96%    59.32%    71.06%
  Recession (−1σ)        5.92%    14.73%    25.66%    38.11%    56.94%    79.14%    88.40%
  Bad (−2σ)             10.37%    23.66%    38.69%    53.65%    73.57%    91.72%    96.68%
  Severe (−3σ)          16.85%    34.96%    53.14%    68.65%    86.11%    97.51%    99.33%


In [51]:
n_quad = 64
gh_nodes, gh_weights = get_gh_nodes_weights(n_quad)

def joint_cdf_mle(d_vec, n_vec, pd_vec, rho_vec, nodes, weights):
    """
    Evaluate joint CDF P(D_1 ≤ d_1, ..., D_K ≤ d_K) via Gauss-Hermite quadrature.
    """
    K = len(d_vec)
    integral = 0.0
    for q in range(len(nodes)):
        x = nodes[q]
        w = weights[q]
        prod = 1.0
        for k in range(K):
            cpd = cond_default_prob(rho_vec[k], pd_vec[k], x)
            prod *= stats.binom.cdf(d_vec[k], n_vec[k], cpd)
        integral += w * prod
    return float(integral)

# Expected defaults (unconditional mean)
d_expected = np.array([int(n_k[k] * pds[k]) for k in range(K)])

# Stressed defaults at X = -4 (extreme stress)
d_stressed = np.array([
    int(n_k[k] * cond_default_prob(rho_mle_final[k], pds[k], -4.0))
    for k in range(K)
])

cdf_at_expected = joint_cdf_mle(d_expected, n_k, pds, rho_mle_final, gh_nodes, gh_weights)
cdf_at_stressed = joint_cdf_mle(d_stressed, n_k, pds, rho_mle_final, gh_nodes, gh_weights)

print("Joint CDF Evaluation [MLE Loadings, 64-pt Gauss-Hermite]")
print(f"\nTotal obligors: {n_k.sum():,}")
print(f"\nCluster sizes: {dict(zip(grades, n_k))}")
print(f"\nExpected defaults (mean scenario): {dict(zip(grades, d_expected))}")
print(f"  → P(D₁ ≤ d₁, ..., D₇ ≤ d₇) = {cdf_at_expected:.6f}")
print(f"\nStressed defaults (X = −4 scenario): {dict(zip(grades, d_stressed))}")
print(f"  → P(D₁ ≤ d₁, ..., D₇ ≤ d₇) = {cdf_at_stressed:.6f}")

Joint CDF Evaluation [MLE Loadings, 64-pt Gauss-Hermite]

Total obligors: 1,957,056

Cluster sizes: {'A': np.int64(416518), 'B': np.int64(603344), 'C': np.int64(552225), 'D': np.int64(255531), 'E': np.int64(96073), 'F': np.int64(26206), 'G': np.int64(7159)}

Expected defaults (mean scenario): {'A': np.int64(15535), 'B': np.int64(57449), 'C': np.int64(93359), 'D': np.int64(66025), 'E': np.int64(38365), 'F': np.int64(15225), 'G': np.int64(4868)}
  → P(D₁ ≤ d₁, ..., D₇ ≤ d₇) = 0.499048

Stressed defaults (X = −4 scenario): {'A': np.int64(30243), 'B': np.int64(108010), 'C': np.int64(172713), 'D': np.int64(116458), 'E': np.int64(64296), 'F': np.int64(23379), 'G': np.int64(6842)}
  → P(D₁ ≤ d₁, ..., D₇ ≤ d₇) = 0.999965


> The joint probability that all 7 clusters simultaneously exceed their stressed default thresholds is extremely low (≈ 0.0035% at $X=-4$). This indicates limited tail dependence across clusters and suggests that systemic default clustering is modest in this portfolio

---
## 5. Monte Carlo Simulation & Stress Testing

Now we simulate from the joint distribution using Monte Carlo. The algorithm goes like:

1. Draw $M$ samples of $X \sim N(0,1)$
2. For each draw $X^{(m)}$, compute $p_k(X^{(m)})$ for all clusters
3. Draw $D_k^{(m)} \sim \text{Bin}(n_k, p_k(X^{(m)}))$ — conditionally independent
4. Compute portfolio loss: $L^{(m)} = \sum_k D_k^{(m)} \cdot LGD_k \cdot EAD_k$

We can then stress test by conditioning on $X \leq x^*$ (i.e., only looking at recession scenarios).

In [52]:
M = 50_000  

ead_k = grade_stats['avg_loan'].values   # EAD per loan by cluster (aligned with grades)
lgd = 0.60                               # assumed LGD

print(f"Simulation Parameters")
print(f"  Monte Carlo paths: {M:,}")
print(f"  LGD:               {lgd:.0%}")
print(f"  Portfolio size:    {n_k.sum():,} obligors")
print(f"  Cluster sizes:     {dict(zip(grades, n_k))}")

# Systematic factor draws
X_draws = np.random.standard_normal(M)

cpd_matrix = np.zeros((M, K))
for k in range(K):
    cpd_matrix[:, k] = conditional_pd_mle(X_draws, pds[k], sqrt_rho_mle_final[k])

default_counts = np.random.binomial(n_k, cpd_matrix)  # shape of (M, K)

# Portfolio-level losses
losses = (default_counts * lgd * ead_k).sum(axis=1)

# Portfolio default rate
total_obligors = n_k.sum()
portfolio_default_rate = default_counts.sum(axis=1) / total_obligors

print(f"\nSimulation Results ({M:,} paths)")
print(f"  Portfolio default rate: mean = {portfolio_default_rate.mean():.2%}, "
      f"std = {portfolio_default_rate.std():.2%}")
print(f"  Loss distribution:     mean = ${losses.mean():,.0f}, "
      f"std = ${losses.std():,.0f}")

var_99 = np.percentile(losses, 99)
var_999 = np.percentile(losses, 99.9)
es_99 = losses[losses >= var_99].mean()
es_999 = losses[losses >= var_999].mean()

print(f"\nRisk Metrics")
print(f"  VaR 99%:   ${var_99:,.0f}")
print(f"  VaR 99.9%: ${var_999:,.0f}")
print(f"  ES 99%:    ${es_99:,.0f}")
print(f"  ES 99.9%:  ${es_999:,.0f}")

Simulation Parameters
  Monte Carlo paths: 50,000
  LGD:               60%
  Portfolio size:    1,957,056 obligors
  Cluster sizes:     {'A': np.int64(416518), 'B': np.int64(603344), 'C': np.int64(552225), 'D': np.int64(255531), 'E': np.int64(96073), 'F': np.int64(26206), 'G': np.int64(7159)}

Simulation Results (50,000 paths)
  Portfolio default rate: mean = 14.82%, std = 7.38%
  Loss distribution:     mean = $2,714,812,866, std = $1,333,426,320

Risk Metrics
  VaR 99%:   $6,552,217,111
  VaR 99.9%: $8,134,605,387
  ES 99%:    $7,233,789,361
  ES 99.9%:  $8,558,528,989


In [53]:
# Risk metrics
var_95 = np.percentile(losses, 95)
var_99 = np.percentile(losses, 99)
var_999 = np.percentile(losses, 99.9)

cvar_95 = losses[losses >= var_95].mean()
cvar_99 = losses[losses >= var_99].mean()
cvar_999 = losses[losses >= var_999].mean()

el = losses.mean()

print("Portfolio Risk Measures")
print(f"  Expected Loss (EL):    ${el:>12,.0f}")
print(f"  VaR 95%:               ${var_95:>12,.0f}")
print(f"  VaR 99%:               ${var_99:>12,.0f}")
print(f"  VaR 99.9%:             ${var_999:>12,.0f}")
print(f"  CVaR/ES 95%:           ${cvar_95:>12,.0f}")
print(f"  CVaR/ES 99%:           ${cvar_99:>12,.0f}")
print(f"  CVaR/ES 99.9%:         ${cvar_999:>12,.0f}")
print(f"  Economic Capital 99%:  ${var_99 - el:>12,.0f}  (VaR99 − EL)")
print(f"  Economic Capital 99.9%: ${var_999 - el:>12,.0f}  (VaR99.9 − EL)")

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=losses, nbinsx=150, name='Loss Distribution',
    marker_color='rgba(52, 152, 219, 0.6)',
    histnorm='probability density'
))

lines = [
    (el,       'EL',        '#2ecc71', 'dash',   'top'),
    (var_95,   'VaR 95%',   '#f39c12', 'dash',   'bottom'),
    (var_99,   'VaR 99%',   '#e74c3c', 'solid',  'top'),
    (var_999,  'VaR 99.9%', '#8e44ad', 'dot',    'bottom'),
    (cvar_99,  'ES 99%',    '#e74c3c', 'dashdot','top'),
]

for val, name, color, dash, pos in lines:
    fig.add_vline(x=val, line_dash=dash, line_color=color,
                  annotation_text=name, annotation_position=f"top right")

fig.update_layout(
    title="Simulated Portfolio Loss Distribution",
    xaxis_title="Portfolio Loss ($)",
    yaxis_title="Density",
    template="plotly_white", height=500,
    showlegend=False
)

fig.add_annotation(
    x=0.98, y=0.90, xref="paper", yref="paper",
    text=f"<b>Risk Summary</b><br>EL: ${el:,.0f}<br>VaR 99%: ${var_99:,.0f}<br>ES 99%: ${cvar_99:,.0f}<br>EC 99%: ${var_99 - el:,.0f}",
    showarrow=False,
    font=dict(size=11, family="Courier New"),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="#333",
    borderwidth=1,
    align="right"
)

fig.show()

Portfolio Risk Measures
  Expected Loss (EL):    $2,714,812,866
  VaR 95%:               $5,186,881,328
  VaR 99%:               $6,552,217,111
  VaR 99.9%:             $8,134,605,387
  CVaR/ES 95%:           $6,012,297,298
  CVaR/ES 99%:           $7,233,789,361
  CVaR/ES 99.9%:         $8,558,528,989
  Economic Capital 99%:  $3,837,404,244  (VaR99 − EL)
  Economic Capital 99.9%: $5,419,792,521  (VaR99.9 − EL)


### Stress Testing the Latent Factor - Recession scenario


We condition on $X \leq x^*$ for various stress levels and examine the resulting loss distribution

In [54]:
stress_levels = {
    'Mild (X ≤ −1)':   (-np.inf, -1),
    'Moderate (X ≤ −1.5)': (-np.inf, -1.5),
    'Severe (X ≤ −2)':     (-np.inf, -2),
    'Extreme (X ≤ −3)':    (-np.inf, -3),
}

stress_colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

print("Stress Test Results")
print(f"{'Scenario':<25s} {'# Paths':>10s} {'Mean Loss':>12s} {'VaR 99%':>12s} "
      f"{'ES 99%':>12s} {'Default Rate':>14s}")
print("-" * 90)

fig = go.Figure()

for (name, (x_lo, x_hi)), color in zip(stress_levels.items(), stress_colors):
    mask = (X_draws >= x_lo) & (X_draws <= x_hi)
    n_paths = mask.sum()
    
    if n_paths < 10:
        print(f"  {name:<23s} -- insufficient paths ({n_paths}) --")
        continue
        
    stressed_losses = losses[mask]
    stressed_dr = portfolio_default_rate[mask]
    
    var_99 = np.percentile(stressed_losses, 99)
    es_99 = stressed_losses[stressed_losses >= var_99].mean()

    print(f"  {name:<23s} {n_paths:>10,d} ${stressed_losses.mean():>11,.0f} "
          f"${var_99:>11,.0f} ${es_99:>11,.0f} {stressed_dr.mean():>14.2%}")

    fig.add_trace(go.Histogram(
        x=stressed_losses, name=f"{name} (n={n_paths:,})", 
        opacity=0.4, marker_color=color, 
        histnorm='probability density', nbinsx=80
    ))

print("-" * 90)
print(f"Note: Unconditional EL = ${losses.mean():,.0f}, VaR 99% = ${np.percentile(losses, 99):,.0f}")

fig.update_layout(
    title="Portfolio Loss Distribution Under Stress Scenarios",
    xaxis_title="Portfolio Loss ($)",
    yaxis_title="Density",
    barmode='overlay',
    template="plotly_white", height=500,
    legend=dict(x=0.55, y=0.95)
)
fig.show()

Stress Test Results
Scenario                     # Paths    Mean Loss      VaR 99%       ES 99%   Default Rate
------------------------------------------------------------------------------------------
  Mild (X ≤ −1)                7,858 $5,006,708,580 $7,802,357,470 $8,334,347,569         27.55%
  Moderate (X ≤ −1.5)          3,272 $5,787,292,090 $8,320,283,997 $8,723,364,555         31.95%
  Severe (X ≤ −2)              1,087 $6,671,881,024 $8,745,147,257 $9,124,968,894         36.95%
  Extreme (X ≤ −3)                63 $8,453,087,404 $9,458,321,899 $9,607,605,111         47.10%
------------------------------------------------------------------------------------------
Note: Unconditional EL = $2,714,812,866, VaR 99% = $6,552,217,111


---
## 6. Conversion of CDF to Survival Function 

In [55]:
print("Default Rate by Cluster Under Stress Scenarios")
print(f"{'Scenario':<25s} {'Paths':>8s}" + "".join(f"{'Grade ' + g:>10s}" for g in grades))


# Unconditional baseline
row = f"  {'Unconditional':<23s} {M:>8,d}"
for k in range(K):
    dr = default_counts[:, k].mean() / n_k[k]
    row += f"{dr:>10.2%}"
print(row)

# Stress scenarios
for name, (x_lo, x_hi) in stress_levels.items():
    mask = (X_draws >= x_lo) & (X_draws <= x_hi)
    n_paths = mask.sum()
    if n_paths < 10:
        print(f"  {name:<23s} -- insufficient paths ({n_paths}) --")
        continue
    row = f"  {name:<23s} {n_paths:>8,d}"
    for k in range(K):
        dr = default_counts[mask, k].mean() / n_k[k]  
        row += f"{dr:>10.2%}"
    print(row)

Default Rate by Cluster Under Stress Scenarios
Scenario                     Paths   Grade A   Grade B   Grade C   Grade D   Grade E   Grade F   Grade G
  Unconditional             50,000     3.71%     9.49%    16.85%    25.77%    39.84%    57.99%    67.89%
  Mild (X ≤ −1)              7,858     8.19%    19.29%    32.34%    46.13%    65.63%    85.93%    93.01%
  Moderate (X ≤ −1.5)        3,272    10.16%    23.15%    37.84%    52.54%    72.23%    90.62%    95.97%
  Severe (X ≤ −2)            1,087    12.68%    27.79%    44.11%    59.42%    78.62%    94.28%    97.93%
  Extreme (X ≤ −3)              63    18.78%    37.99%    56.65%    71.92%    88.35%    98.17%    99.57%


The impact of stress is convex: default rates rise sharply for high-quality grades but level off for lower-quality grades. Since F and G loans already have elevated baseline default rates (50–almost 70%), they approach a saturation point where more economic deterioration has diminishing marginal impact

In [56]:
def joint_survival_mle(d_vec, n_vec, pd_vec, rho_vec, nodes, weights):
    """
    Evaluate joint survival function P(D_1 > d_1, ..., D_K > d_K)
    via Gauss-Hermite quadrature.
    """
    K = len(d_vec)
    integral = 0.0
    for q in range(len(nodes)):
        x = nodes[q]
        w = weights[q]
        prod = 1.0
        for k in range(K):
            cpd = cond_default_prob(rho_vec[k], pd_vec[k], x)
            # P(D_k > d_k) = 1 - P(D_k ≤ d_k)
            surv_k = 1.0 - stats.binom.cdf(d_vec[k], n_vec[k], cpd)
            prod *= surv_k
        integral += w * prod
    return float(integral)

# Survival probability of exceeding expected defaults
surv_expected = joint_survival_mle(d_expected, n_k, pds, rho_mle_final, gh_nodes, gh_weights)
print(f"P(D_1 > d_1, ..., D_7 > d_7) at expected thresholds = {surv_expected:.6f}")

# Survival probability of exceeding stressed defaults
surv_stressed = joint_survival_mle(d_stressed, n_k, pds, rho_mle_final, gh_nodes, gh_weights)
print(f"P(D_1 > d_1, ..., D_7 > d_7) at stressed thresholds = {surv_stressed:.6f}")

P(D_1 > d_1, ..., D_7 > d_7) at expected thresholds = 0.499622
P(D_1 > d_1, ..., D_7 > d_7) at stressed thresholds = 0.000034


Under stress, the odds of every cluster staying below its stressed default level drop to nearly zero — meaning at least one cluster will almost certainly go past its stress threshold

In [57]:
# Compute 99% VaR thresholds per cluster from simulation
var_99_per_cluster = np.percentile(default_counts, 99, axis=0).astype(int)

surv_joint_tail = joint_survival_mle(
    var_99_per_cluster, n_k, pds, rho_mle_final, gh_nodes, gh_weights
)
print(f"Joint tail survival probability: {surv_joint_tail:.6e}")

Joint tail survival probability: 4.388793e-14


In [58]:
# survival at different quantile thresholds
quantiles = [0.50, 0.75, 0.90, 0.95, 0.99]
threshold_matrix = np.zeros((len(quantiles), K), dtype=int)

for i, q in enumerate(quantiles):
    for k in range(K):
        threshold_matrix[i, k] = np.percentile(default_counts[:, k], q * 100)

print("Joint Survival Probability at Different Thresholds")
print(f"{'Quantile':<12s}", end="")
for g in grades:
    print(f"{'Grade ' + g:>12s}", end="")
print(f"{'Joint Survival':>18s}")

for i, q in enumerate(quantiles):
    surv = joint_survival_mle(threshold_matrix[i], n_k, pds, rho_mle_final, gh_nodes, gh_weights)
    print(f"{q:.0%} threshold ", end="")
    for k in range(K):
        print(f"{threshold_matrix[i, k]:>12,d}", end="")
    print(f"{surv:>18.6e}")

Joint Survival Probability at Different Thresholds
Quantile         Grade A     Grade B     Grade C     Grade D     Grade E     Grade F     Grade G    Joint Survival
50% threshold       13,005      50,499      84,691      61,901      37,350      15,509       5,078      4.102361e-01
75% threshold       20,193      74,610     121,187      85,041      49,015      19,216       5,999      5.682558e-02
90% threshold       29,002     102,040     160,207     108,125      59,359      21,849       6,552      1.274253e-04
95% threshold       35,376     120,842     185,557     122,095      65,103      23,053       6,764      1.728830e-07
99% threshold       50,934     163,154     238,856     149,971      75,050      24,691       7,013      4.388793e-14


Some observations:
1. The joint survival probability declines by several orders of magnitude as the threshold increases from the 50th to the 99th percentile.

2. The near-zero probability at the 99th percentile means negligible joint tail dependence across clusters.

3. The results show that idiosyncratic variation dominates portfolio behavior. Simultaneous outperformance—or simultaneous extreme underperformance—across all clusters is statistically implausible, so there is effective risk dispersion across grades.

## Survival Copula for Clustered Defaults

The joint survival function can be expressed via the **survival Gaussian copula**:

$$\bar{C}(u_1, \ldots, u_K) = P(U_1 > u_1, \ldots, U_K > u_K)$$

For the one-factor Gaussian copula:

$$\bar{C}(u_1, \ldots, u_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \left[ 1 - \Phi\!\left(\frac{\Phi^{-1}(u_k) - \sqrt{\rho_k}\, x}{\sqrt{1 - \rho_k}}\right) \right] \phi(x)\, dx$$

So by Sklar's theorem, the joint survival function of default counts is:

$$P(D_1 > d_1, \ldots, D_K > d_K) = \bar{C}\!\big(F_1(d_1), \ldots, F_K(d_K)\big)$$

where $F_k$ is the marginal CDF of defaults in cluster $k$.

In [59]:
scenario_data = {
    'Unconditional': (X_draws >= -np.inf) & (X_draws <= np.inf), 
    'Mild (X ≤ −1)': X_draws <= -1,
    'Moderate (X ≤ −1.5)': X_draws <= -1.5,
    'Severe (X ≤ −2)': X_draws <= -2,
    'Extreme (X ≤ −3)': X_draws <= -3,
}

scenario_names = list(scenario_data.keys())
dr_matrix = np.zeros((len(scenario_names), K))

for i, (name, mask) in enumerate(scenario_data.items()):
    n_paths = mask.sum()
    if n_paths < 10:
        print(f"Warning: {name} has only {n_paths} paths")
    for k in range(K):
        dr_matrix[i, k] = default_counts[mask, k].mean() / n_k[k]

fig = go.Figure(data=go.Heatmap(
    z=dr_matrix,
    x=[f'Grade {g}' for g in grades],
    y=scenario_names,
    colorscale='RdYlGn_r',
    text=[[f'{val:.2%}' for val in row] for row in dr_matrix],
    texttemplate='%{text}',
    textfont={"size": 12},
    colorbar_title="Default Rate"
))

fig.update_layout(
    title="Default Rate Heatmap Across Stress Scenarios",
    xaxis_title="Risk Grade",
    yaxis_title="Scenario",
    template="plotly_white",
    height=400
)
fig.show()

In [60]:
def expected_shortfall_counts(d_vec, n_vec, pd_vec, rho_vec, nodes, weights, M_es=10000):
    """
    Monte Carlo estimate of E[D_k | D_1 > d_1, ..., D_K > d_K] for each cluster.
    """
    # Importance sampling or simple rejection sampling
    X_draws = np.random.standard_normal(M_es * 10)  # oversample
    cpd_matrix = np.zeros((len(X_draws), K))
    for k in range(K):
        cpd_matrix[:, k] = cond_default_prob(rho_vec[k], pd_vec[k], X_draws)
    
    counts = np.random.binomial(n_vec, cpd_matrix)
    
    # Filter to joint exceedances
    exceed_mask = np.all(counts > d_vec, axis=1)
    if exceed_mask.sum() < 100:
        return None, None
    
    es_counts = counts[exceed_mask].mean(axis=0)
    n_exceedances = exceed_mask.sum()
    return es_counts, n_exceedances

es_counts, n_ex = expected_shortfall_counts(d_expected, n_k, pds, rho_mle_final, gh_nodes, gh_weights)

if es_counts is not None:
    print(f"Expected Shortfall of Defaults (given all clusters exceed expected)")
    print(f"  Based on {n_ex:,} joint exceedance scenarios")
    for k, g in enumerate(grades):
        print(f"  Grade {g}: E[D_k | exceed] = {es_counts[k]:,.0f} "
              f"(vs unconditional E[D_k] = {d_expected[k]:,.0f}, "
              f"ratio = {es_counts[k]/d_expected[k]:.2f}x)")

Expected Shortfall of Defaults (given all clusters exceed expected)
  Based on 46,402 joint exceedance scenarios
  Grade A: E[D_k | exceed] = 17,966 (vs unconditional E[D_k] = 15,535, ratio = 1.16x)
  Grade B: E[D_k | exceed] = 66,168 (vs unconditional E[D_k] = 57,449, ratio = 1.15x)
  Grade C: E[D_k | exceed] = 107,625 (vs unconditional E[D_k] = 93,359, ratio = 1.15x)
  Grade D: E[D_k | exceed] = 75,651 (vs unconditional E[D_k] = 66,025, ratio = 1.15x)
  Grade E: E[D_k | exceed] = 43,881 (vs unconditional E[D_k] = 38,365, ratio = 1.14x)
  Grade F: E[D_k | exceed] = 17,433 (vs unconditional E[D_k] = 15,225, ratio = 1.15x)
  Grade G: E[D_k | exceed] = 5,511 (vs unconditional E[D_k] = 4,868, ratio = 1.13x)


In [61]:
def marginal_survival_contribution(k, d_vec, n_vec, pd_vec, rho_vec, nodes, weights):
    """Survival probability when only cluster k is constrained."""
    K = len(d_vec)
    integral = 0.0
    for q in range(len(nodes)):
        x = nodes[q]
        w = weights[q]
        prod = 1.0
        for j in range(K):
            cpd = cond_default_prob(rho_vec[j], pd_vec[j], x)
            if j == k:
                prod *= (1.0 - stats.binom.cdf(d_vec[j], n_vec[j], cpd))
        integral += w * prod
    return float(integral)

print("Marginal Survival Contribution by Cluster")
print(f"{'Grade':<8s} {'P(D_k > d_k)':>16s} {'Joint (all)':>16s}")
joint = joint_survival_mle(d_expected, n_k, pds, rho_mle_final, gh_nodes, gh_weights)
for k, g in enumerate(grades):
    marg = marginal_survival_contribution(k, d_expected, n_k, pds, rho_mle_final, gh_nodes, gh_weights)
    print(f"{g:<8s} {marg:>16.6f} {joint:>16.6f}")

Marginal Survival Contribution by Cluster
Grade        P(D_k > d_k)      Joint (all)
A                0.499622         0.499622
B                0.500000         0.499622
C                0.500000         0.499622
D                0.500000         0.499622
E                0.500000         0.499622
F                0.500000         0.499622
G                0.500952         0.499622


In [62]:
rho_multipliers = [1.0, 1.5, 2.0, 3.0, 5.0]
print("Joint Survival Sensitivity to Asset Correlation")
print(f"{'rho Multiplier':<14s} {'Joint Survival':>18s}")

for mult in rho_multipliers:
    rho_stressed = np.minimum(rho_mle_final * mult, 0.99)
    surv = joint_survival_mle(d_expected, n_k, pds, rho_stressed, gh_nodes, gh_weights)
    print(f"{mult:>6.1f}x         {surv:>18.6e}")

Joint Survival Sensitivity to Asset Correlation
rho Multiplier     Joint Survival
   1.0x               4.996216e-01
   1.5x               4.997757e-01
   2.0x               4.997782e-01
   3.0x               4.994724e-01
   5.0x               4.902043e-01
